In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D 

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

def construct_model(
  arch,
  train_embed_layer,
  act,
  width,
  hid_n_layers,
):
  # arch : "MLP", "ResNet", "DiffNet-Add", "DiffNet-Mul"
  # train_embed_layer : bool, whether to train the first layer
  # act : "tanh", "sin", "relu", "gelu"
  # width : int, number of hidden units in each layer
  # hid_n_layers : int, number of hidden layers
  if arch == "MLP":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=False
    )
  elif arch == "ResNet":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=True
    )
  elif arch == "DiffNet-Add":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='add'
    )
  elif arch == "DiffNet-Mul":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='mul'
    )
  elif arch.startswith("DiffNet-Const"):
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='const',
      scale_const=float(arch.split('-')[-1]) if '-' in arch else 1.0
    )

class MLP(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    residual=False
  ):
    super(MLP, self).__init__()
    self.train_embed_layer = train_embed_layer
    self.width = width
    self.hid_n_layers = hid_n_layers
    self.residual = residual

    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin

    self.embed_layer = nn.Linear(2, width, bias=True)
    if not self.train_embed_layer:
      self.embed_layer.weight.requires_grad = False
      self.embed_layer.bias.requires_grad = False
    
    self.hidden_layers = nn.ModuleList()
    for _ in range(hid_n_layers - 1):
      self.hidden_layers.append(nn.Linear(width, width, bias=True))
    
    self.output_layer = nn.Linear(width, 1, bias=True)
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    x = self.embed_layer(combined_input)
    x = torch.sin(x) 
    for layer in self.hidden_layers:
      h = layer(x)
      if self.residual:
        x = self.act(h) + x
      else:
        x = self.act(h)
    x = self.output_layer(x)
    return x  
class DiffNet(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    scale_type='add',
    scale_const=1.0,
  ):
    super(DiffNet, self).__init__()

    self.begin_embed_layer = nn.Linear(2, width, bias=True)
    if not train_embed_layer:
      self.begin_embed_layer.weight.requires_grad = False
      self.begin_embed_layer.bias.requires_grad = False
    
    self.embed_layers = nn.ModuleList()
    self.hidden_layers = nn.ModuleList()
    for i in range(hid_n_layers):
      embed_layer = nn.Linear(2, width, bias=True)
      if scale_type == 'add':
        embed_layer.weight.data *= (i + 2)
        embed_layer.bias.data *= (i + 2)
      elif scale_type == 'mul':
        embed_layer.weight.data *= 2**(i + 1)
        embed_layer.bias.data *= 2**(i + 1)
      elif scale_type == 'const' and scale_const != 1.0:
        embed_layer.weight.data *= scale_const
        embed_layer.bias.data *= scale_const
      
      if not train_embed_layer:
        embed_layer.weight.requires_grad = False
        embed_layer.bias.requires_grad = False
      self.embed_layers.append(embed_layer)

      self.hidden_layers.append(nn.Linear(2 * width, width, bias=True))
    
    self.output_layer = nn.Linear(width, 1, bias=True)

    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    x = self.begin_embed_layer(combined_input)
    x = torch.sin(x)
    for embed_layer, hidden_layer in zip(self.embed_layers, self.hidden_layers):
      h = embed_layer(combined_input)
      h = torch.sin(h)

      z = torch.cat((x, h), dim=-1)
      z = hidden_layer(z)
      x = self.act(z) + x

    x = self.output_layer(x)
    return x + 5


In [ ]:
class ConstrainedSigmoid(torch.autograd.Function):
  """
  A custom autograd function to find a scalar `c` for each batch element
  such that mean_i(sigmoid(y_i + c)) = p, and to compute its custom gradient.

  Forward Pass:
  - For each `y` in the batch, it finds a `c` using the Newton-Raphson method.

  Backward Pass:
  - Computes the gradient of the loss with respect to `y` using the
    implicit function theorem.
  """

  @staticmethod
  def forward(ctx, y, p, num_iterations=20, tolerance=1e-6, mode='newton-rhapson'):
    batch_size = y.shape[0]
    if mode == 'newton-rhapson':
      c = torch.full((batch_size, 1), np.log(p / (1.0 - p)), device=y.device, dtype=y.dtype) - torch.mean(y, dim=1, keepdim=True)
      minimax = torch.max(y, dim=1, keepdim=True).values - torch.min(y, dim=1, keepdim=True).values

      for i in range(num_iterations):
        s = torch.sigmoid(y + c)
        f_c = torch.mean(s, dim=1, keepdim=True) - p
        
        if torch.max(torch.abs(f_c)) < tolerance:
          break

        s_prime = s * (1 - s)
        f_prime_c = torch.mean(s_prime, dim=1, keepdim=True)
        
        c = c - torch.clamp(f_c / (f_prime_c + 1e-8), min=-minimax / (i + 1), max=minimax / (i + 1))

      c_final = c.squeeze(1)
    elif mode == 'bisection':
      lb = torch.full((batch_size, 1), np.log(p / (1.0 - p)), device=y.device, dtype=y.dtype) - torch.max(y, dim=1, keepdim=True).values
      ub = torch.full((batch_size, 1), np.log(p / (1.0 - p)), device=y.device, dtype=y.dtype) - torch.min(y, dim=1, keepdim=True).values

      for _ in range(num_iterations):
        midpoint = (lb + ub) / 2.0
        s = torch.sigmoid(y + midpoint)
        f_mid = torch.mean(s, dim=1, keepdim=True) - p

        if torch.max(torch.abs(f_mid)) < tolerance:
          c_final = midpoint.squeeze(1)
          break

        sign = torch.sign(f_mid)
        lb = torch.where(sign < 0, midpoint, lb)
        ub = torch.where(sign > 0, midpoint, ub)
      
      c_final = (lb + ub) / 2.0
    else:
      raise ValueError("Unsupported mode. Use 'newton-rhapson' or 'bisection'.")
    ctx.save_for_backward(y, c_final)
    return c_final

  @staticmethod
  def backward(ctx, grad_output):
    y, c = ctx.saved_tensors

    c = c.unsqueeze(1)

    s = torch.sigmoid(y + c)
    s_prime = s * (1 - s)

    denominator = torch.sum(s_prime, dim=1, keepdim=True)

    dc_dy = -s_prime / (denominator + 1e-8) # Add epsilon for stability

    grad_y = grad_output.unsqueeze(1) * dc_dy

    return grad_y, None, None, None

constrained_sigmoid = ConstrainedSigmoid.apply

In [ ]:
first_coordinates =  [0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
second_coordinates = [1, 2, 3, 4, 0, 5, 6, 7, 8, 9, 7, 8, 9, 5, 6]

perm_indices = [
  [0, 28, 9, 3, 24, 21, 34, 40, 41, 36, 17, 43, 4, 13, 37],
  [0, 28, 21, 12, 40, 11, 5, 17, 43, 1, 33, 38, 41, 32, 25],
  [6, 28, 9, 2, 31, 15, 43, 32, 1, 20, 35, 33, 38, 16, 25],
  [18, 27, 15, 2, 31, 17, 23, 14, 39, 35, 4, 33, 38, 7, 16],
  [18, 24, 37, 31, 8, 5, 23, 14, 39, 4, 42, 10, 16, 25, 22],
  [10, 6, 18, 3, 24, 12, 5, 14, 44, 29, 35, 19, 41, 42, 22],
  [0, 6, 12, 34, 39, 30, 44, 20, 29, 26, 7, 19, 10, 32, 22],
  [34, 27, 15, 8, 11, 36, 1, 30, 20, 29, 26, 13, 7, 19, 42],
  [9, 3, 21, 37, 27, 40, 2, 8, 11, 36, 23, 30, 44, 26, 13]
]

def calculate_max_objective_expectation_with_constraints(model, num_samples, target, device):
  points = torch.rand(num_samples, 10, device=device)

  indices = torch.combinations(torch.arange(10, device=device), r=2)

  pairs_first = points[:, indices[:, 0]]
  pairs_second = points[:, indices[:, 1]]

  new_x = torch.minimum(pairs_first, pairs_second)  # Take minimum of the two sets of coordinates
  new_y = torch.maximum(pairs_first, pairs_second)  # Take maximum of the two sets of coordinates

  f_xy = model(new_x, new_y) 

  shift = constrained_sigmoid(f_xy.reshape(1, -1), target) 
  f_xy = torch.sigmoid(f_xy + shift).squeeze(-1) 

  res = f_xy[:, perm_indices] 
  
  return torch.mean(torch.prod(res, dim=2)), shift, f_xy
  

In [ ]:
from tqdm import tqdm
def bootstrap_ci_torch_batched(data, num_batches=1000, batch_size=100):
  n_size = data.numel()
  n_bootstrap = num_batches * batch_size
  
  all_bootstrap_means = torch.zeros(n_bootstrap, device=data.device, dtype=data.dtype)

  for i in tqdm(range(num_batches), desc="Processing Batches"):
    indices = torch.randint(0, n_size, (batch_size, n_size), device=data.device)
    
    bootstrap_samples = data[indices]

    batch_means = bootstrap_samples.mean(dim=1)
    
    start_index = i * batch_size
    end_index = start_index + batch_size
    all_bootstrap_means[start_index:end_index] = batch_means

  lower_bound = torch.quantile(all_bootstrap_means, 0.025)
  upper_bound = torch.quantile(all_bootstrap_means, 0.975)

  return (lower_bound.item(), upper_bound.item())

In [ ]:
import os

batch_size = 2 ** 16

for file in os.listdir():
  if file.startswith("best_model_lb_"):
    file_split = file[:-3].split("_")
    
    target = int(file_split[3]) / 100
    if target in [50, 75]:
      continue
    arch = file_split[4]
    act = file_split[5]
    width = int(file_split[6])
    depth = int(file_split[7])
    lr = float(file_split[8])

    model = construct_model(
      arch=arch,
      train_embed_layer=False,
      act=act,
      width=width,
      hid_n_layers=depth
    ).to(DEVICE)

    print(f"Loading {arch}-{act}-w{width}-d{depth}-lr{lr} for target {target}")

    model.load_state_dict(torch.load(file, map_location=DEVICE))

    model.eval()

    with torch.no_grad():
      obj, result = estimate_max_objective_expectation_with_constraints(model, batch_size, 2 ** 12, target, DEVICE)
      print(f"Estimated objective: {obj.item():8e}")
      ci = bootstrap_ci_torch_batched(result.flatten().to(DEVICE), num_batches=10000, batch_size=1)
      print(f"95% CI: [{ci[0]:8e}, {ci[1]:8e}]")
